# Análise de orçamento de Google Ads Automatizada com IA

O código abaixo consiste em usar LLMs via prompt para acelerar tarefas de análise.

O objetivo é usar dados reais do Google Ads, enviar para uma IA via código, e receber análise automática completa da evolução do investimento de acordo com o orçamento definido, identificando subutilização de orçamento, oscilações entre dias e campanhas que merecem atenção. Vamos seguir o seguinte passo a passo:



*   Carregar dados reais de campanha;
*   Tratar e selecionar as colunas relevantes;
*   Montar um prompt com contexto de negócio;
*   Enviar para uma IA via código;
*   Receber uma análise automática em linguagem natural.

## Bloco 1 - Instalar a biblioteca do Gemini

In [ ]:
!pip install -q google-genai

A nova biblioteca ``google-genai`` usa nomes de modelo diferentes. Vamos primeiro descobrir quais modelos estão disponíveis na conta, usando o comando abaixo:

In [ ]:
from google import genai

client = genai.Client(api_key="MINHA-API-KEY-GEMINI")

for modelo in client.models.list():
    print(modelo.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

O que esse bloco fez:




*   ``client = genai.Client(api_key="AIza...")``: Cria uma conexão autenticada com a API do Google. A key é sua identidade, sem ela a API rejeita.


## Bloco 2 - Primeira chamada ao Gemini via código (conexão real com API de IA)

Vamos usar o modelo ``gemini-2.5-flash``. O comando abaixo é apenas para avaliar se o modelo está sendo aceito e rodando.

In [ ]:
resposta = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Olá! Me explique em 2 linhas o que é CPA no marketing digital."
)

print(resposta.text)

CPA (Custo Por Aquisição/Ação) é uma métrica que indica quanto você gasta para obter uma conversão específica, como uma venda, cadastro ou download, no marketing digital. Ajuda a medir a eficiência dos seus investimentos em publicidade.


O que esse bloco fez:

*   ``resposta = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Olá! Me explique..."
)``: o código manda a pergunta para o Gemini pela internet e espera a resposta. model é qual versão usar, contents é a pergunta.

*   ``print(resposta.text)``: A variável resposta guarda um objeto com várias informações. O .text pega só o texto da resposta.

## Bloco 3 - Dados reais em arquivo CSV


### Leitura e tratamento de dados

Vamos usar dados reais sem tratamento no IA.

A abertura do arquivo com ``df = pd.read_csv("gads-4-6-maio-2026.csv", sep=";")`` estava gerando alguns erros.

Usei o comando abaixo para ver se poderia ter a ver com a estrutura do arquivo.

In [ ]:
with open("gads-4-7-maio-2026.csv", "r", encoding="latin1") as f:
    for i, linha in enumerate(f):
        print(f"Linha {i}: {linha[:100]}")
        if i > 1:
            break

Linha 0: RelatÃ³rio de campanha

Linha 1: 4 de maio de 2026 - 7 de maio de 2026

Linha 2: Dia,Status da campanha,Campanha,OrÃ§amento,Nome do orÃ§amento,Tipo de orÃ§amento,Status,Motivos do s


O arquivo tem:

>Linha 0: título "Relatório de campanha"

>Linha 1: período

>Linha 2: cabeçalho real das colunas

>Linha 3 em diante: dados

Vou importar o pandas porque vou começar a precisar fazer um tratamento nos dados.

Vou printar só o cabeçalho para evitar mostrar dados que possam ser sensíveis.

In [ ]:
import pandas as pd

df = pd.read_csv("gads-4-7-maio-2026.csv", encoding="latin1", skiprows=2)
df.head(0)

,Dia,Status da campanha,Campanha,OrÃ§amento,Nome do orÃ§amento,Tipo de orÃ§amento,Status,Motivos do status,PontuaÃ§Ã£o de otimizaÃ§Ã£o,Valor conv. / custo,...,Freq. mÃ©d. impr. / usuÃ¡rio,Freq. mÃ©d. impr. / usuÃ¡rio (30 dias),Metas,ID da campanha,CTR visÃ­vel,CPM mÃ©d. visÃ­vel,Impr. visÃ­veis,AÃ§Ãµes originadas no app,Custo / aÃ§Ã£o originada no app,Valor original da conv.


Agora deu certo porque usamos skiprows=2, que manda o pandas pular as 2 primeiras linhas antes de começar a ler.

Notemos que o arquivo tem 43 colunas, vamos usar só as necessárias. Listando as colunas abaixo:

In [ ]:
print(df.columns.tolist())

['Dia', 'Status da campanha', 'Campanha', 'OrÃ§amento', 'Nome do orÃ§amento', 'Tipo de orÃ§amento', 'Status', 'Motivos do status', 'PontuaÃ§Ã£o de otimizaÃ§Ã£o', 'Valor conv. / custo', 'ROAS desejado', 'ROAS mÃ©d. desejado', 'CÃ³digo da moeda', 'Custo', 'Valor conv.', 'ProporÃ§Ã£o', 'ConversÃµes', 'Custo / conv.', 'CPC desejado', 'Cliques', 'Impr.', 'CTR', 'CPC mÃ©d.', 'Taxa de conv.', 'Tipo de estratÃ©gia de lances', 'CPA desejado', 'CPA mÃ©d. desejado', 'CPA Compra', 'Tipo de campanha', 'Valor da conversÃ£o em dispositivos diferentes', 'Custo / conv. (comparÃ¡vel entre plataformas)', 'Parcela de impressÃµes desejada', 'Parcela impr rede de pesquisa', 'Freq. mÃ©d. impr. / usuÃ¡rio', 'Freq. mÃ©d. impr. / usuÃ¡rio (30 dias)', 'Metas', 'ID da campanha', 'CTR visÃ\xadvel', 'CPM mÃ©d. visÃ\xadvel', 'Impr. visÃ\xadveis', 'AÃ§Ãµes originadas no app', 'Custo / aÃ§Ã£o originada no app', 'Valor original da conv.']


Vou mudar o nome da coluna OrÃ§amento e Tipo de OrÃ§amento por Orçamento e Tipo de Orçamento, respectivamente,, porque, como são variáveis que vou usar, quero seguir com elas certinhas.

In [ ]:
df = df.rename(columns={"OrÃ§amento": "Orçamento", "Tipo de orÃ§amento": "Tipo de orçamento"})
print(df.columns.tolist())

['Dia', 'Status da campanha', 'Campanha', 'Orçamento', 'Nome do orÃ§amento', 'Tipo de orçamento', 'Status', 'Motivos do status', 'PontuaÃ§Ã£o de otimizaÃ§Ã£o', 'Valor conv. / custo', 'ROAS desejado', 'ROAS mÃ©d. desejado', 'CÃ³digo da moeda', 'Custo', 'Valor conv.', 'ProporÃ§Ã£o', 'ConversÃµes', 'Custo / conv.', 'CPC desejado', 'Cliques', 'Impr.', 'CTR', 'CPC mÃ©d.', 'Taxa de conv.', 'Tipo de estratÃ©gia de lances', 'CPA desejado', 'CPA mÃ©d. desejado', 'CPA Compra', 'Tipo de campanha', 'Valor da conversÃ£o em dispositivos diferentes', 'Custo / conv. (comparÃ¡vel entre plataformas)', 'Parcela de impressÃµes desejada', 'Parcela impr rede de pesquisa', 'Freq. mÃ©d. impr. / usuÃ¡rio', 'Freq. mÃ©d. impr. / usuÃ¡rio (30 dias)', 'Metas', 'ID da campanha', 'CTR visÃ\xadvel', 'CPM mÃ©d. visÃ\xadvel', 'Impr. visÃ\xadveis', 'AÃ§Ãµes originadas no app', 'Custo / aÃ§Ã£o originada no app', 'Valor original da conv.']


Vou selecionar só as colunas de Dia, Campanha, Orçamento, Tipo de Orçamento e Custo, pois quero uma avaliação sobre o seguinte problema.

Os dados que estou usando  cobrem os dias 4, 5 e 6 de maio de 2026. No dia 3 de maio foram feitos ajustes de orçamento nas campanhas. Quero analisar se o Custo realizado está seguindo o Orçamento configurado, se houve oscilações relevantes entre os dias e que seja apontado quais campanhas que merecem atenção.

In [ ]:
colunas = ["Dia", "Campanha", "Orçamento", "Custo", "Tipo de orçamento", "Custo / conv."]
df_analise = df[colunas].copy()
df_analise = df_analise[df_analise["Campanha"].notna()]
df_analise = df_analise[df_analise["Dia"].notna()]

df_analise.head(0)

,Dia,Campanha,Orçamento,Custo,Tipo de orçamento,Custo / conv.


Trocando o encoding da palavra Diário antes de enviar para o Gemini.

In [ ]:
df_analise["Tipo de orçamento"] = df_analise["Tipo de orçamento"].str.replace("DiÃ¡rio", "Diário").str.replace("Total da campanha", "Total da campanha")

print(df_analise["Tipo de orçamento"].unique())

['Diário' 'Total da campanha' ' --']


In [ ]:
df_analise.head(0)

,Dia,Campanha,Orçamento,Custo,Tipo de orçamento,Custo / conv.


### Gerando o prompt - Análise automática em Linguagem Natural

Vou disponibilizar apenas o código abaixo para não compartilhar dados estratégicos, mas funciona. O erro que pode dar é se sobrecarga no Gemini.

In [ ]:
tabela_texto = df_analise.to_string(index=False)

prompt = f"""
Você é um analista de mídia sênior especializado em Google Ads.

Contexto: Os dados abaixo cobrem os dias 4 até 7 de maio.
No dia 3 de maio foram feitos ajustes de orçamento nas campanhas.
Hoje ainda é dia 7, os dados de custo desse dia ainda são de hoje.
Análise dos resultados entre os dias 4 a 7 vão ser
relaventes para sugerir possíveis ajustes nas campanhas a serem feitos hoje. Também leve em consideração o Custo/conv.

IMPORTANTE sobre o campo "Tipo de orçamento":
- "Diário": o valor de Orçamento representa o limite diário da campanha.
  Compare diretamente Orçamento vs Custo por dia.
- "Total da campanha":  ignore para fins de comparação direta.
- "--":  ignore para fins de comparação direta.

Analise:
Quais campanhas é mais recomendado fazer ajuste de aumento ou diminuição de orçamento?


DADOS:
{tabela_texto}
"""

resposta = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={"temperature": 0.1} # vou adicionar esse parâmetro para controlar o quanto o LLM 'cria' resposta vs o quanto 'segue um padrão'
)

print(resposta.text)

### Salvando em arquivo .txt

O próximo passo é salvar esse relatório automaticamente num arquivo .txt, para que no futuro o script faça tudo sozinho e entregue o resultado salvo sem precisar copiar nada.

In [ ]:
from datetime import date

nome_arquivo = f"relatorio_{date.today()}.txt"

with open(nome_arquivo, "w", encoding="utf-8") as f:
    f.write(resposta.text)

print(f"Relatório salvo como: {nome_arquivo}")

Relatório salvo como: relatorio_2026-05-07.txt


## Bloco 4 - Script para enviar o relatório por email

Nessa etapa, vou usar o email do Gmail. Será necessário configurar a **senha de app** do Gmail. Isso é uma senha especial criada só para o script usar nesse contexto.

Segue um passo a passo dessa etapa:


*   Acesse myaccount.google.com;
*   Clique em "Segurança" no menu lateral;
*   Em "Como você faz login no Google", clique em "Verificação em duas etapas" e veja se já está ativada. Se não estiver, ativa primeiro;
*   Depois de confirmar que está ativa, volte em Segurança e pesquise "Senhas de app" na barra de busca do próprio Google Account;
*   Em "Nome do app", digite um nome para a senha, pode ser, por exemplo ``script_relatorio`` e clique em "Criar";
*   Vai aparecer uma senha de 16 caracteres; copie ela agora porque ela só aparece uma vez. Ao aplicar essa senha no comando abaixo, insira ela sem espaços.

A senha de app fica vinculada à conta Google que você estiver usando no Colab.

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from datetime import date

EMAIL_REMETENTE = "meuemail@gmail.com"
EMAIL_SENHA     = "senha-de-app-do-gmail"
EMAIL_DESTINO   = ["email-1@gmail.com", "email-2@gmail.com"]

msg = MIMEMultipart()
msg["From"]    = EMAIL_REMETENTE
msg["To"]      = ", ".join(EMAIL_DESTINO)
msg["Subject"] = f"Campanhas (Análise em Linguagem Natural) | {date.today()}"

corpo = "Segue em anexo o relatório automático de análise de campanhas gerado pela API do Gemini."
msg.attach(MIMEText(corpo, "plain"))

with open(nome_arquivo, "rb") as f:
    anexo = MIMEBase("application", "octet-stream")
    anexo.set_payload(f.read())
    encoders.encode_base64(anexo)
    anexo.add_header("Content-Disposition", f"attachment; filename={nome_arquivo}")
    msg.attach(anexo)

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as servidor:
    servidor.login(EMAIL_REMETENTE, EMAIL_SENHA)
    servidor.sendmail(EMAIL_REMETENTE, EMAIL_DESTINO, msg.as_string())

print(f"✅ Email enviado para: {EMAIL_DESTINO}")